# Working with files
Paths, CSV, Excel, writing output and handling errors. Each section matches a slide.

## Paths with pathlib

In [ ]:
from pathlib import Path

DATA = Path("../../data")
print(DATA)
print(DATA.resolve())
print(DATA.exists())

## Joining paths and file details

In [ ]:
path = DATA / "trade_summary.csv"
print(path)
print(path.name)
print(path.suffix, path.stem)
print(path.exists())

## Finding files with glob

In [ ]:
for f in sorted(DATA.glob("*.csv")):
    kb = f.stat().st_size / 1024
    print(f"{f.name:28} {kb:5.1f} KB")

## Looking inside a CSV file

In [ ]:
path = DATA / "tariffs_mfn.csv"
with open(path, encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().rstrip())

## Reading CSV with the csv module

In [ ]:
import csv

path = DATA / "trade_summary.csv"
with open(path, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(len(rows), "rows")
first = rows[0]
print(first["reporter"], first["year"])
print(first["exports_usd_m"])
print(type(first["exports_usd_m"]))

## Reading CSV with pandas

In [ ]:
import pandas as pd

trade = pd.read_csv(path)
print(trade.shape)
trade.dtypes

## read_csv options

In [ ]:
small = pd.read_csv(
    path,
    usecols=["reporter", "year",
             "exports_usd_m"],
    nrows=3)
small

## Reading Excel

In [ ]:
xl_path = DATA / "countries.xlsx"
xl = pd.ExcelFile(xl_path)
print(xl.sheet_names)

countries = pd.read_excel(
    xl_path, sheet_name="countries")
countries[["iso3", "country_name"]].head(3)

## Writing CSV and Excel

In [ ]:
OUT = Path("output")
OUT.mkdir(exist_ok=True)

kenya = trade[trade["reporter"] == "Kenya"]
kenya.to_csv(OUT / "kenya.csv",
             index=False)
kenya.to_excel(OUT / "kenya.xlsx",
               index=False)
print(sorted(p.name for p in OUT.iterdir()))

## Writing a text file

In [ ]:
with open(OUT / "notes.txt", "w",
          encoding="utf-8") as f:
    f.write(f"Rows: {len(kenya)}\n")
    f.write("Source: trade_summary.csv\n")
print((OUT / "notes.txt").read_text())

## An error stops the program

In [ ]:
pd.read_csv(DATA / "not_there.csv")
print("This line never runs")

## Handling errors with try and except

In [ ]:
def load_csv(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print(f"Missing file: {path.name}")
    return None

df = load_csv(DATA / "not_there.csv")
print(df is None)
print(len(load_csv(DATA / "tariffs_mfn.csv")))